# <center>Estimating large scale matching models</center>
### <center>Alfred Galichon (NYU & Sciences Po), Antoine Jacquet (Sciences Po) and Georgy Salakhutdinov (Polytechnique)</center>
## <center>'math+econ+code' masterclass series, Collegio Carlo Alberto, March 2026</center>
#### <center>With python code examples</center>
© 2018–2026 by Alfred Galichon. Past and present support from NSF grant DMS-1716489, ERC grant CoG-866274 are acknowledged, as well as inputs from contributors listed [here](http://www.math-econ-code.org/team).

**If you reuse material from this masterclass, please cite as:**<br>
Alfred Galichon, 'math+econ+code' masterclass series. https://www.math-econ-code.org/

## Learning objectives

* Method of moments estimation for matching with linearly parametrized surplus
* Simulated method of moments for general heterogeneity
* Probit vs. Logit estimation


## References

* Choo & Siow (2006). Who marries whom and why. *Journal of Political Economy*
* Galichon & Salanié (2022). Cupid’s Invisible Hand: Social Surplus and Identification in Matching Models. *Review of Economic Studies*
* Galichon, Jacquet, Salakhutdinov (2025). Transferable Utility Matching Beyond Logit: Computation and Estimation with General Heterogeneity. *Working paper*, https://arxiv.org/abs/2511.23116


## Libraries


In [1]:
import numpy as np
import pandas as pd
import gurobipy as grb


# Estimation for large scale matching models

## Setting

The previous lectures have focused on the assignment problem; that is, given a surplus matrix $\Phi$, how to find a matching which maximizes the total surplus in the population.
We have also seen that linear duality yields a competitive equilibrium interpretation of the same problem.

Now we turn to the *estimation problem*; that is, given an observed matching, how to recover the values of the surplus $\Phi$.  
Recall that under the separability assumption:
* the surplus from match $ij$ can be decomposed as $\tilde \Phi_{ij} = \Phi_{xy} + \varepsilon_{iy} + \eta_{xj}$,
* singlehood yields payoff $\varepsilon_{i0}$ for woman $i$ and $\eta_{0j}$ for man $j$,
* the idiosyncratic preference shocks $\varepsilon_{iy}$ and $\eta_{xj}$ follow conditional distributions
$\varepsilon_{iy} \, | \, x_i = x \; \sim \; \mathbf P_x$ and $\eta_{xj} \, | \, y_j = y \; \sim \; \mathbf Q_y$.

We assume that we observe a matrix $\hat \mu = (\hat \mu_{xy})$ of aggregated empirical matches by type, and we hope to recover the values of the systematic surpluses $\Phi_{xy}$.  
We'll consider that the surplus $\Phi_{xy}$ can be linearly parametrized by a known basis of functions $(\phi_{xyk})$, i.e.
\begin{equation}
    \Phi_{xy} = \sum_{k=1}^K \phi_{xyk} \lambda_k, \qquad \text{or in matrix form,} 
    \qquad \Phi = \phi \lambda.
\end{equation}
Our goal is therefore to recover the values $\lambda_k$.

We will see that this problem can in fact be solved with techniques very similar to those we saw for the assignment problem.


## Method of moments estimator

Let's go back to the assignment problem, whose value we now consider explicitly as a function of $\Phi$:
\begin{align}
\mathcal W(\Phi) = \max_{\pi_{iy}, \pi_{xj} \geq 0} ~ & \sum_i \bigg[ \pi_{i0} \varepsilon_{i0} + \sum_{y \in Y} \pi_{iy} \Big( \overbrace{\sum_{x \in X} \delta_{ix} \frac{\Phi_{xy}}{2} + \varepsilon_{iy}}^{= \alpha_{iy}} \Big) \bigg]
+ \sum_j \bigg[ \pi_{0j} \eta_{0j} + \sum_{x \in X} \pi_{xj} \Big( \overbrace{\sum_{y \in Y} \delta_{jy} \frac{\Phi_{xy}}{2} + \eta_{xj}}^{= \gamma_{xj}} \Big) \bigg] \\
\text{s.t.} ~ & \textstyle\sum_i \delta_{ix} \pi_{iy} = \textstyle\sum_j \delta_{jy} \pi_{xj} \\
              & \pi_{i0} + \textstyle\sum_{y \in Y} \pi_{iy} = 1 \\
              & \pi_{0j} + \textstyle\sum_{x \in X} \pi_{xj} = 1.
\end{align}

Denote $\pi^*$ a solution to this problem, and call $\mu^*$ the matrix of associated aggregate number of matches by type, i.e. $\mu_{xy}^* = \textstyle\sum_i \delta_{ix} \pi_{iy}^* = \textstyle\sum_j \delta_{jy} \pi_{xj}^*$.  
The envelope theorem applied to the function $\mathcal W (\Phi)$ yields

\begin{align}
\frac{\partial \mathcal W}{\partial \Phi_{xy}} (\Phi)
&= \sum_i \pi_{iy}^* \delta_{ix} \frac{1}{2} + \sum_j \pi_{xj}^* \delta_{jy} \frac{1}{2} \\
&= \frac{1}{2} (\mu_{xy}^* + \mu_{xy}^*) = \mu_{xy}^*
\end{align}
hence $\nabla \mathcal W (\Phi) = \mu^*$.

Taking into account our linear parametrization $\Phi = \phi \lambda$ we thus get
\begin{equation}
\phi^\top \nabla \mathcal W(\phi \lambda) = \phi^\top \mu^*.
\end{equation}

This equation suggests a natural **method of moments estimator** for $\lambda$: it is the value $\hat \lambda$ such that

\begin{equation}
\boxed{\phi^\top \mu^* (\hat \lambda) = \phi^\top \hat \mu.}
\end{equation}

Galichon & Salanié (2022) observed that this equation is simply the first-order condition of the following optimization problem:
\begin{equation}
    \min_\lambda ~ \mathcal W (\phi \lambda) - (\phi \lambda)^\top \hat \mu.
\end{equation}

But we can simplify this problem even further:
by explicitly writing the maximization program for $\mathcal W (\phi \lambda)$ and switching the min and the max, the parameters $\lambda_k$ end up playing the role of dual variables for new constraints which are exactly the moment-matching conditions:


---

**Proposition.** (Galichon, Jacquet, Salakhutdinov 2025)  
The simulated moment-matching estimator $\hat \lambda = (\hat \lambda_k)$ is obtained as the vector of Lagrange multipliers of the constraints indexed by $k$ in the following linear program:
\begin{align}
\max_{\pi_{iy}, \pi_{i0}, \pi_{xj}, \pi_{0j} \geq 0} ~&
\sum_i \Big( \pi_{i0} \varepsilon_{i0} + \sum_y \pi_{iy} \varepsilon_{iy} \Big) + \sum_j \Big( \pi_{0j} \eta_{0j} + \sum_x \pi_{xj} \eta_{xj} \Big) \tag{$E$} \\
\text{s.t.} ~ & \pi_{i0} + \textstyle \sum_y \pi_{iy} = 1 \qquad (\forall i) \notag \\
              & \pi_{0j} + \textstyle \sum_x \pi_{xj} = 1 \qquad (\forall j) \notag \\
              & \textstyle \sum_i \delta_{ix} \pi_{iy} = \sum_j \delta_{jy} \pi_{xj} \qquad (\forall xy) \notag \\
              & \textstyle \sum_{xy} \frac{1}{2} \big(\sum_i \delta_{ix} \pi_{iy} + \sum_j \delta_{jy} \pi_{xj} \big) \phi_{xyk} = \sum_{xy} \hat \mu_{xy} \phi_{xyk} \qquad (\forall k). \notag
\end{align}

---

The problem $(E)$ is extremely similar to the assignment problem we have already encountered: instead of featuring the surplus terms in its objective, it imposes extra moment-matching constraints on the surplus.

## Algorithm

Notice that actually, we have one problem left:
we do not observe the preference shocks $\varepsilon_{iy}$ and $\eta_{xj}$, so we cannot directly compute the program above.

However, when the size of the population is very large compared to the sizes of $X$ and $Y$, simulations will give a good enough approximation of the distributions $\mathbf P_x$ and $\mathbf Q_y$.
We thus draw the $\varepsilon_{iy}$ and $\eta_{xj}$.

Lastly, we recognize that the program above is actually an assignment problem with extra moment-matching constraints! This can also be solved using an algorithm similar to the one we saw for the assignment problem, which is an application of Dantzig–Wolfe.


---

**Algorithm: Dantzig–Wolfe for simulated method of moments estimation**

*Step 0.*  
For all $x$, simulate a sample $(\varepsilon_{iy})$ of size $n_x$ from $\mathbf P_x$;
and for all $y$, simulate a sample $(\eta_{xj})$ of size $m_y$ from $\mathbf Q_y$.
Initialize the choice sets as $Y_i^0 = \emptyset$ for $i \in I$ and $X_j^0 = \emptyset$ for $j \in J$. Go to step 1.

*Step t.*  
1. Solve the restricted assignment problem with moment-matching constraints $(E)$ using only variables in the current choice sets $Y_i = Y_i^t$ and $X_j = X_j^t$. Obtain a restricted optimal matching $\pi^t$ and its vectors of Lagrange multipliers $u^t, v^t, w^t, \lambda^t$.
2. If
\begin{align*}
&u_i^t \geq \max_{y \in Y \setminus Y_i} ~ \sum_x \delta_{ix} \Big(\frac{1}{2} \sum_k \phi_{xyk} \lambda_k^t - w_{xy}^t \Big) + \varepsilon_{iy} \quad (\forall i) \\
\text{and} \quad
&v_j^t \geq \max_{x \in X \setminus X_j} ~ \sum_y \delta_{jy} \Big(\frac{1}{2} \sum_k \phi_{xyk} \lambda_k^t + w_{xy}^t\Big) + \eta_{xj} \quad (\forall j),
\end{align*}
stop.
Otherwise, go to 3.  
3. For each $i \in I$, if $i$ does not satisfy her above inequality, pick a type $y_i$ in the argmax and define $Y_i^{t+1} = Y_i^t \cup \{y_i\}$; otherwise, define $Y_i^{t+1} = Y_i^t$.
Similarly, for each $j \in J$, if $j$ does not satisfy his above inequality, pick a type $x_j$ in the argmax and define $X_j^{t+1} = X_j^t \cup \{x_j\}$; otherwise, define $X_j^{t+1} = X_j^t$.
Go to step $t+1$.

---


## Computation

We apply this algorithm on the data Choo & Siow (2006).

We begin by loading the data.
We also use custom functions from the math+econ+code package which are essentially based on the Dantzig—Wolfe code developed in Lecture 2.

In [2]:
#pip install mec
#!pip install --upgrade mec

from mec.dw import Dantzig_Wolfe, Dantzig_Wolfe_vector_approach, Dantzig_Wolfe_Estimation

thepath = 'https://raw.githubusercontent.com/math-econ-code/mec_optim_2021-01/master/data_mec_optim/marriage-ChooSiow/'

n_singles = pd.read_csv(thepath+'n_singles.txt', sep='\t', header = None)
marr = pd.read_csv(thepath+'marr.txt', sep='\t', header = None)

def prepareChooSiow_data(n_singles,marr,startCateg=0,endCateg=35):
    μhat_x0 = np.array(n_singles[0].iloc[startCateg:endCateg])
    μhat_0y = np.array(n_singles[1].iloc[startCateg:endCateg])
    μhat_x_y = np.array(marr.iloc[startCateg:endCateg,startCateg:endCateg])
    Nhat = 2 * μhat_x_y.sum() + μhat_x0.sum() + μhat_0y.sum()    

    return μhat_x_y, μhat_x0, μhat_0y

μhat_x_y, μhat_x0, μhat_0y = prepareChooSiow_data(n_singles, marr)

scale = 10e-4
μhat_x_y = (μhat_x_y * scale).astype(int)
μhat_x0 = (μhat_x0 * scale).astype(int)
μhat_0y = (μhat_0y * scale).astype(int)

print('Total number of individuals in the subsample = '+str(2*μhat_x_y.sum() + μhat_x0.sum() + μhat_0y.sum())+'.')
print('Total number of women = '+str(μhat_x_y.sum() + μhat_x0.sum())+'.')
print('Total number of men = '+str(μhat_x_y.sum() + μhat_0y.sum())+'.')


Total number of individuals in the subsample = 16184.
Total number of women = 8242.
Total number of men = 7942.


We will consider two models:
1. errors follow i.i.d. Gumbel distributions,
2. errors follow i.i.d. $\mathcal N(0,1)$ distributions.

Let's start with the Gumbel model.


We begin by choosing our explanatory variables for the surplus.
If $x$ is the woman's age and $y$ is the man's age, we parametrize the systematic surplus $\Phi_{xy}$ as a quadratic function of the partner's ages:
\begin{equation}
\Phi_{xy} = \lambda_0 + \lambda_1 \, x + \lambda_2 \, y + \lambda_3 \, x^2 + \lambda_4 \, y^2 + \lambda_5 \, xy.
\end{equation}

We build our matrix of regressors `phi_x_y_k`.

In [3]:
I = μhat_x_y.sum() + μhat_x0.sum()
J = μhat_x_y.sum() + μhat_0y.sum()
X, Y = μhat_x_y.shape

K = 6   # we use 6 explanatory variables : constant, x, y, x^2, y^2, x*y

x = np.arange(X)
y = np.arange(Y)

phi_x_y_k = np.zeros((X,Y,K))
phi_x_y_k[:,:,0] = 1
phi_x_y_k[:,:,1] = x.reshape(-1,1)
phi_x_y_k[:,:,2] = y
phi_x_y_k[:,:,3] = (x**2).reshape(-1,1)
phi_x_y_k[:,:,4] = y**2
phi_x_y_k[:,:,5] = x.reshape(-1,1) * y


# with Kronecker products:

# phi_xy_k = np.column_stack([np.ones(X*Y),             # 1
#                             np.kron(x,np.ones(Y)),    # x
#                             np.kron(np.ones(X),y),    # y
#                             np.kron(x**2,np.ones(Y)), # x**2
#                             np.kron(np.ones(X),y**2), # y**2
#                             np.kron(x,y)              # x * y
#                             ])

# phi_x_y_k_kron = phi_xy_k.reshape((X,Y,K))

# np.max((phi_x_y_k - phi_x_y_k_kron)**2)

In [4]:
# Probit estimation
estim_Probit = Dantzig_Wolfe_Estimation(μhat_x_y, μhat_x0, μhat_0y, phi_x_y_k,
    np.random.normal(0,1,I), np.random.normal(0,1,J), np.random.normal(0,1,(I,Y)), np.random.normal(0,1,(X,J)))
estim_Probit.column_generation()
estim_Probit.get_lambdas()

Restricted license - for non-production use only - expires 2027-11-29


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [5]:
# Logit estimation
beta = np.sqrt(6)/np.pi;  mu_g = -0.5772*beta
estim_Logit = Dantzig_Wolfe_Estimation(μhat_x_y, μhat_x0, μhat_0y, phi_x_y_k,
    np.random.gumbel(mu_g,beta,I), np.random.gumbel(mu_g,beta,J), np.random.gumbel(mu_g,beta,(I,Y)), np.random.gumbel(mu_g,beta,(X,J)))
estim_Logit.column_generation()
estim_Logit.get_lambdas()

GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information